In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS bronze')
spark.sql('SHOW SCHEMAS').show()

In [0]:
def ingest_raw_to_bronze(tableName):
    raw_table = spark.table(tableName)
    raw_table = raw_table.withColumn('_ingestion_timestamp', F.current_timestamp())
    raw_table = raw_table.withColumn('_source', F.lit(tableName))
    raw_table.write.format('delta').mode('overwrite').saveAsTable('bronze.'+tableName.replace('_raw',''))
    print(f'Complete ingestion table {tableName}')

tables = spark.sql('show tables')
tablename_list = [tab.tableName for tab in tables.collect()]
for table in tablename_list:
    ingest_raw_to_bronze(table)

In [0]:
for t in tablename_list:
    source_count  = spark.table(t).count()
    bronze_count = spark.table('bronze.' + t.replace('_raw','')).count()
    print(
        f"{t}: {source_count:,} "
        f"→ {'bronze.' + t.replace('_raw','')}: {bronze_count:,}"
    )

In [0]:
spark.sql('show tables in bronze').show()